# RetailIQ – Smart Inventory Intelligence & Predictive Analytics Platform
### End-to-End Machine Learning, Demand Forecasting & Inventory Optimization Notebook

**Author:** Senior Machine Learning Engineer & Data Scientist  
**Objective:** Build an intelligent inventory analytics platform that models sales patterns, predicts future SKU demand, identifies stockout risks, clusters products and customers, detects anomalies, and generates prescriptive inventory actions.

## 1. Setup Environment and Imports

In [ ]:
import os
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, generate_synthetic_retail_data
from src.preprocessing import clean_retail_data, calculate_eda_summary
from src.feature_engineering import (
    build_daily_product_demand_features,
    build_stockout_classification_features,
    build_product_clustering_features,
    build_customer_rfm_features
)
from src.demand_forecasting import train_demand_model, forecast_future_demand
from src.stockout_prediction import train_stockout_model, predict_stockout_risk_scenario
from src.product_clustering import train_product_clustering
from src.customer_segmentation import train_customer_segmentation
from src.anomaly_detection import train_anomaly_model
from src.inventory_engine import evaluate_product_inventory_status

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("All RetailIQ modules successfully loaded!")

## 2. Load and Inspect Dataset

In [ ]:
# Load raw dataset
df_raw = load_dataset('../data/raw/retail_inventory.csv')
df_clean = clean_retail_data(df_raw)
eda = calculate_eda_summary(df_clean)

print(f"Total Records: {len(df_clean):,}")
print(f"Unique Products: {eda['total_products']}")
print(f"Date Range: {eda['date_range_start']} to {eda['date_range_end']}")
print(f"Total Revenue: ${eda['total_revenue']:,.2f}")
print(f"Average Profit Margin: {eda['avg_profit_margin_pct']:.1f}%")
df_clean.head()

## 3. Exploratory Data Analysis (EDA)
Let us visualize revenue trends, category contributions, and sales distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Category Revenue
cat_rev = df_clean.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
cat_rev.plot(kind='bar', ax=axes[0], color='#0284C7')
axes[0].set_title('Total Revenue by Category ($)', fontsize=14)
axes[0].set_ylabel('Revenue ($)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

# Monthly Revenue Trend
df_clean['Sales Date'] = pd.to_datetime(df_clean['Sales Date'])
monthly = df_clean.set_index('Sales Date').resample('M')['Revenue'].sum()
monthly.plot(ax=axes[1], color='#10B981', marker='o', linewidth=2)
axes[1].set_title('Monthly Revenue Momentum', fontsize=14)
axes[1].set_ylabel('Revenue ($)')

plt.tight_layout()
plt.show()

## 4. Machine Learning Model 1: Random Forest Demand Forecasting

In [ ]:
demand_bundle = train_demand_model(df_clean, save_path='../models/demand_model.pkl')
print("Demand Model Metrics:", demand_bundle['metrics'])

# Top feature importances
demand_bundle['feature_importances'].head(8)

## 5. Machine Learning Model 2: Random Forest Stockout Risk Classification

In [ ]:
stockout_bundle = train_stockout_model(df_clean, save_path='../models/stockout_model.pkl')
print("Stockout Risk Model Metrics:", stockout_bundle['metrics'])
stockout_bundle['feature_importances'].head(8)

## 6. Machine Learning Model 3 & 4: K-Means Clustering
- Multidimensional Product Segmentation
- Customer RFM Segmentation

In [ ]:
prod_bundle = train_product_clustering(df_clean, save_path='../models/product_cluster_model.pkl')
cust_bundle = train_customer_segmentation(df_clean, save_path='../models/customer_cluster_model.pkl')

print("Product Clustering Silhouette Score:", prod_bundle['final_silhouette'])
print("Customer Clustering Silhouette Score:", cust_bundle['final_silhouette'])

## 7. Machine Learning Model 5: Isolation Forest Anomaly Detection

In [ ]:
anomaly_bundle = train_anomaly_model(df_clean, save_path='../models/anomaly_model.pkl')
print("Anomaly Summary:", anomaly_bundle['summary_stats'])
anomaly_bundle['anomalies_df'][['Sales Date', 'Product ID', 'Units Sold', 'Anomaly_Score', 'Anomaly_Reason']].head(5)

## 8. Prescriptive Inventory Intelligence Engine
Calculates Reorder Points, Safety Stock, EOQ, and actionable replenishment plans.

In [ ]:
inventory_plan = evaluate_product_inventory_status(df_clean, stockout_bundle)
inventory_plan[[
    'Product ID', 'Product_Name', 'Current_Stock', 'Calculated_Reorder_Point',
    'Stock_Cover_Days', 'ML_Stockout_Risk', 'Primary_Recommendation', 'Suggested_Reorder_Qty'
]].head(10)